### **AES: Password-Based Encryption, CBC Malleability, and GCM**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jrafa1607/Cryptography_In_Python/blob/main/Algorithms/AES_Cipher.ipynb)

> **AES** is the standard symmetric cipher. Using it safely depends on the details: deriving the key from a password with a slow **KDF**, using a fresh random **IV/nonce**, and **authenticating** the ciphertext. This notebook encrypts a file with a password, shows a **bit-flipping attack** on unauthenticated AES-CBC, and fixes it with **AES-GCM**.

#### **01. Key Derivation from a Password**

A password is not a key. **PBKDF2-HMAC-SHA256** stretches it with a random salt and many iterations, making each guess expensive. OWASP recommends **600,000 iterations** for PBKDF2-HMAC-SHA256 (the original notebook used 100,000). Memory-hard functions such as scrypt or Argon2 are even better against GPU attacks.

In [1]:
# Derive a 256-bit key from a password
import os
import time

from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC

ITERATIONS = 600_000

def derive_key(password: str, salt: bytes) -> bytes:
    kdf = PBKDF2HMAC(algorithm=hashes.SHA256(), length=32, salt=salt, iterations=ITERATIONS)
    return kdf.derive(password.encode())

salt = os.urandom(16)
start = time.perf_counter()
key = derive_key("correct horse battery staple", salt)
print(f"Derived key: {key.hex()}")
print(f"Time per guess: {time.perf_counter() - start:.2f} s  (this slowness is what protects weak passwords)")

Derived key: e64d56738c3fd3e8955885cd962a221112d92e0ba60785068a2cb0e0bdce6367
Time per guess: 0.09 s  (this slowness is what protects weak passwords)


#### **02. Unauthenticated AES-CBC Is Malleable**

CBC hides the content, but **does not detect changes**. Flipping a bit in the IV flips the same bit in the first plaintext block, so an attacker who knows the format can change `amount=0100` to `amount=9100` without the key. This is also the root of **padding oracle** attacks.

In [2]:
# Encrypt with AES-CBC, then change the plaintext by editing only the ciphertext
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.primitives.padding import PKCS7

def cbc_encrypt(key, plaintext):
    iv = os.urandom(16)
    padder = PKCS7(128).padder()
    enc = Cipher(algorithms.AES(key), modes.CBC(iv)).encryptor()
    return iv + enc.update(padder.update(plaintext) + padder.finalize()) + enc.finalize()

def cbc_decrypt(key, data):
    dec = Cipher(algorithms.AES(key), modes.CBC(data[:16])).decryptor()
    unpadder = PKCS7(128).unpadder()
    return unpadder.update(dec.update(data[16:]) + dec.finalize()) + unpadder.finalize()

message = b"amount=0100;to=alice;memo=rent"
token = bytearray(cbc_encrypt(key, message))

position = message.index(b"0100")                        # the attacker knows the message format
token[position] ^= ord("0") ^ ord("9")                    # flip bits in the IV only
print("Original :", cbc_decrypt(key, bytes(cbc_encrypt(key, message))))
print("Tampered :", cbc_decrypt(key, bytes(token)), "<- accepted, no error")

Original : b'amount=0100;to=alice;memo=rent'
Tampered : b'amount=9100;to=alice;memo=rent' <- accepted, no error


#### **03. AES-GCM: Encryption plus Integrity**

**AES-GCM** is an **AEAD** mode (authenticated encryption with associated data): it encrypts and adds a 16-byte authentication tag. Any change to the ciphertext, the nonce, or the associated data makes decryption fail. The nonce must **never repeat** under the same key.

In [3]:
# Same attack against AES-GCM: the change is detected
from cryptography.exceptions import InvalidTag
from cryptography.hazmat.primitives.ciphers.aead import AESGCM

aead = AESGCM(key)
nonce = os.urandom(12)
header = b"invoice-v1"                       # authenticated but not encrypted
sealed = bytearray(aead.encrypt(nonce, message, header))
print("Decrypted:", aead.decrypt(nonce, bytes(sealed), header))

sealed[position] ^= ord("0") ^ ord("9")
try:
    aead.decrypt(nonce, bytes(sealed), header)
except InvalidTag:
    print("Tampered : InvalidTag, the message was rejected")

Decrypted: b'amount=0100;to=alice;memo=rent'
Tampered : InvalidTag, the message was rejected


#### **04. Encrypting a File with a Password**

The file format stores everything needed to decrypt except the password: `salt (16) | nonce (12) | ciphertext + tag`.

In [4]:
# Password-based file encryption with PBKDF2 + AES-GCM
import tempfile
from pathlib import Path

def encrypt_file(path: Path, password: str) -> Path:
    salt, nonce = os.urandom(16), os.urandom(12)
    sealed = AESGCM(derive_key(password, salt)).encrypt(nonce, path.read_bytes(), None)
    out = path.with_suffix(path.suffix + ".enc")
    out.write_bytes(salt + nonce + sealed)
    return out

def decrypt_file(path: Path, password: str) -> bytes:
    data = path.read_bytes()
    return AESGCM(derive_key(password, data[:16])).decrypt(data[16:28], data[28:], None)

with tempfile.TemporaryDirectory() as folder:
    original = Path(folder) / "report.txt"
    original.write_text("Quarterly results: confidential.")
    encrypted = encrypt_file(original, "correct horse battery staple")
    print("Encrypted bytes:", encrypted.read_bytes()[:40].hex(), "...")
    print("Decrypted      :", decrypt_file(encrypted, "correct horse battery staple").decode())
    try:
        decrypt_file(encrypted, "wrong password")
    except InvalidTag:
        print("Wrong password : InvalidTag (no partial or garbled output)")

Encrypted bytes: 583ea5d043fe986a49e2f4c7956c6157476a1d1865f959413dc9fe38eb5acc535651b7a5e510a7ab ...
Decrypted      : Quarterly results: confidential.
Wrong password : InvalidTag (no partial or garbled output)


#### **Key Takeaways**

* Derive keys from passwords with a slow KDF and a random salt.
* Encryption without authentication lets attackers modify data; prefer **AES-GCM** or ChaCha20-Poly1305.
* Never reuse a nonce with the same key, and show ciphertext as hex or Base64, never as decoded text.